# Week 3 — Gradient Descent and Automatic Differentiation
**80-minute computer seminar.** Complete every `TODO`; use optimizer libraries only for the final reference check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
np.random.seed(7)
plt.rcParams.update({'figure.figsize': (6, 4), 'axes.grid': True})

## 1. Fixed-step gradient descent
Implement the optimizer and return enough history to diagnose convergence. Use the gradient test together with iteration and finite-value safeguards.

In [ ]:
def gradient_descent(f, grad, x0, step, tol=1e-8, max_iter=10_000):
    x = np.asarray(x0, dtype=float).copy()
    history = {'x': [x.copy()], 'f': [float(f(x))], 'grad_norm': [], 'step': []}
    for _ in range(max_iter):
        g = np.asarray(grad(x), dtype=float)
        gn = np.linalg.norm(g)
        history['grad_norm'].append(gn)
        if not np.isfinite(gn):
            raise FloatingPointError('non-finite gradient')
        if gn <= tol:
            break
        # TODO: form x_new and add a non-finite objective check
        x_new = None
        raise NotImplementedError('complete the fixed-step update')
    return x, history

### Quadratic test problem
For $f(x)=\tfrac12x^TQx-b^Tx$, $L=\lambda_{\max}(Q)$ and $\mu=\lambda_{\min}(Q)$.

In [ ]:
def make_quadratic(Q, b=None):
    Q = np.asarray(Q, dtype=float)
    b = np.zeros(Q.shape[0]) if b is None else np.asarray(b, dtype=float)
    f = lambda x: 0.5 * x @ Q @ x - b @ x
    grad = lambda x: Q @ x - b
    x_star = np.linalg.solve(Q, b)
    return f, grad, x_star

Q = np.diag([1.0, 10.0])
f, grad, x_star = make_quadratic(Q)
L, mu = np.linalg.eigvalsh(Q)[[-1, 0]]
print(f'L={L:g}, mu={mu:g}, kappa={L/mu:g}, stability boundary={2/L:g}')

## 2. Stability experiment
Compare $\alpha\in\{0.05,0.19,0.20,0.21\}$. Plot objective error on a log scale and explain the result in eigen-coordinates.

In [ ]:
steps = [0.05, 0.19, 0.20, 0.21]
x0 = np.array([2.0, 2.0])
# TODO: run a bounded number of iterations for each step and plot f(x_k)-f*
# Keep divergent runs finite by stopping when ||x|| exceeds a stated threshold.


## 3. Well- and poorly-conditioned trajectories
Compare $Q_1=\mathrm{diag}(1,2)$ and $Q_2=\mathrm{diag}(1,50)$. Overlay contours and iterates; report predicted and observed contraction.

In [ ]:
def plot_contours_and_path(Q, path, ax, title):
    path = np.asarray(path)
    lim = 1.1 * np.max(np.abs(path))
    grid = np.linspace(-lim, lim, 250)
    X, Y = np.meshgrid(grid, grid)
    Z = 0.5 * (Q[0,0]*X**2 + 2*Q[0,1]*X*Y + Q[1,1]*Y**2)
    ax.contour(X, Y, Z, levels=18, cmap='Blues')
    ax.plot(path[:,0], path[:,1], 'o-', ms=3, color='tab:orange')
    ax.set(title=title, xlabel='$x_1$', ylabel='$x_2$', aspect='equal')

# TODO: run GD with alpha=1/L on both problems and create a two-panel figure.


## 4. Armijo backtracking
Do not recompute the gradient in the trial loop. Reject non-descent directions and cap the number of reductions.

In [ ]:
def backtracking(f, x, g, d, alpha0=1.0, beta=0.5, c=1e-4, max_shrinks=60):
    if g @ d >= 0:
        raise ValueError('d must be a strict descent direction')
    alpha = float(alpha0)
    fx = float(f(x))
    for n_shrinks in range(max_shrinks + 1):
        # TODO: test the Armijo inequality and return (alpha, n_shrinks)
        pass
    raise RuntimeError('line search failed')

def gd_armijo(f, grad, x0, tol=1e-8, max_iter=10_000):
    # TODO: combine backtracking with GD; record accepted steps and function calls.
    raise NotImplementedError

## 5. Stopping criteria
Run gradient norm, step norm, and function-change tests separately. Find a case where a change-based rule stops too early, then propose a combined, scale-aware rule.

In [ ]:
def stopping_report(f_old, f_new, x_old, x_new, g_new, atol=1e-8, rtol=1e-6):
    return {
        'gradient': np.linalg.norm(g_new) <= atol + rtol * max(1.0, abs(f_new)),
        'step': np.linalg.norm(x_new-x_old) <= atol + rtol * max(1.0, np.linalg.norm(x_old)),
        'function': abs(f_new-f_old) <= atol + rtol * max(1.0, abs(f_old)),
    }
# TODO: demonstrate premature stopping caused by a tiny prescribed step.


## 6. A computational graph by hand
For $z=\sin(xy)+x^2$ at $(x,y)=(2,-1)$, list nodes in topological order, compute their values, and propagate adjoints backward.

In [ ]:
x, y = 2.0, -1.0
u = x * y
v = np.sin(u)
z = v + x**2
# TODO: compute bar_z, bar_v, bar_u, bar_x, and bar_y explicitly.
print('forward values:', {'x': x, 'y': y, 'u': u, 'v': v, 'z': z})

## 7. Scalar reverse-mode automatic differentiation
Complete the missing operations. Every local backward closure must **accumulate** into parent gradients with `+=`.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f'Value(data={self.data:.6g}, grad={self.grad:.6g})'

    @staticmethod
    def _coerce(other):
        return other if isinstance(other, Value) else Value(other)

    def __add__(self, other):
        other = self._coerce(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = self._coerce(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # TODO: write both local derivative contributions using +=
            raise NotImplementedError
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        # TODO: implement a numeric power and its local backward rule
        raise NotImplementedError

    def exp(self):
        # TODO
        raise NotImplementedError

    def log(self):
        # TODO: reject non-positive data
        raise NotImplementedError

    def tanh(self):
        # TODO
        raise NotImplementedError

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        for v in topo:
            v.grad = 0.0
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    __radd__ = __add__
    __rmul__ = __mul__
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-self._coerce(other))
    def __rsub__(self, other): return self._coerce(other) + (-self)
    def __truediv__(self, other): return self * self._coerce(other)**-1
    def __rtruediv__(self, other): return self._coerce(other) * self**-1


### Shared-path test
After completing `Value`, this must return $2x+3x^2=16$ at $x=2$. An assignment bug instead of accumulation will fail.

In [ ]:
# x = Value(2.0)
# z = x**2 + x**3
# z.backward()
# print(z, x.grad)
# assert np.isclose(x.grad, 16.0)


## 8. Gradient checking
Compare AD with central differences for $h=10^{-1},\ldots,10^{-10}$. Use relative error $|g_{AD}-g_{FD}|/\max(1,|g_{AD}|,|g_{FD}|)$.

In [ ]:
def central_difference(fun, x, h=1e-6):
    x = np.asarray(x, dtype=float)
    g = np.empty_like(x)
    for i in range(x.size):
        e = np.zeros_like(x); e[i] = 1.0
        g[i] = (fun(x+h*e) - fun(x-h*e)) / (2*h)
    return g

# TODO: choose a composite expression supported by Value, compare gradients, and plot error vs h.


## 9. Logistic regression with your optimizer
Implement stable loss using `np.logaddexp(0, scores) - y*scores`. Train with both fixed-step and Armijo GD.

In [ ]:
rng = np.random.default_rng(7)
n, p = 300, 3
X = rng.normal(size=(n, p))
w_true = np.array([1.5, -2.0, 0.7])
prob = 1 / (1 + np.exp(-(X @ w_true)))
y = rng.binomial(1, prob).astype(float)

def logistic_loss(w):
    s = X @ w
    return np.mean(np.logaddexp(0.0, s) - y*s)

def logistic_grad(w):
    s = X @ w
    # Stable sigmoid, written without SciPy.
    p_hat = np.where(s >= 0, 1/(1+np.exp(-s)), np.exp(s)/(1+np.exp(s)))
    return X.T @ (p_hat-y) / len(y)

# TODO: estimate or upper-bound L, train with fixed alpha and Armijo, and compare histories.
# TODO: report final loss, gradient norm, iterations, and classification accuracy.


## 10. Reference check and reflection
Optionally compare your final parameters with `scipy.optimize.minimize`. Explain: (1) why $2/L$ is a boundary rather than a safe default; (2) how conditioning changes the path; (3) why reverse-mode gradients accumulate.

In [ ]:
# Optional reference check only after your implementation works:
# from scipy.optimize import minimize
# reference = minimize(logistic_loss, np.zeros(p), jac=logistic_grad, method='BFGS')
# print(reference.fun, np.linalg.norm(reference.jac), reference.x)
